In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load Data
df = pd.read_csv("substance_survey.csv")

# Select Features + Target
features = [
    "nicotine_score",
    "nicotine_risk",
    "audit_score",
    "audit_risk",
    "drug_mar",
    "aca_substance_impact_count",
    "sub_effort_binary",
    "sub_knowwhere_rev"
]

target = "grade_composite"

df_model = df[features + [target]]

# Drop rows where target is missing
df_model = df_model.dropna(subset=[target])

X = df_model[features]
y = df_model[target]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Random Forest
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        random_state=42
    ))
])

# Train Model
rf_pipeline.fit(X_train, y_train)

# Evaluate
def evaluate(model, X_test, y_test, name):
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    print(f"\n{name}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R^2: {r2:.3f}")

evaluate(rf_pipeline, X_test, y_test, "Random Forest")

# Feature Importance
importances = rf_pipeline.named_steps["model"].feature_importances_

feat_imp = pd.DataFrame({
    "feature": features,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print("\nFeature Importances:")
print(feat_imp)


Random Forest
RMSE: 0.577
R^2: -0.004

Feature Importances:
                      feature  importance
2                 audit_score    0.366452
7           sub_knowwhere_rev    0.253998
0              nicotine_score    0.157390
5  aca_substance_impact_count    0.137862
6           sub_effort_binary    0.073545
3                  audit_risk    0.010753
1               nicotine_risk    0.000000
4                    drug_mar    0.000000


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("substance_survey.csv")

features = [
    "nicotine_score",
    "audit_score",
    "drug_mar",
    "aca_substance_impact_count",
    "sub_effort_binary",
    "sub_knowwhere_rev"
]

target = "grade_composite"

df_model = df[features + [target]].dropna(subset=[target])

X = df_model[features]
y = df_model[target]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Impute missing values
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R²:", r2)

# Coefficients
coef_df = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_
})

print("\nCoefficients:")
print(coef_df)

RMSE: 0.57543670603411
R²: 0.002338783418806556

Coefficients:
                      feature   coefficient
0              nicotine_score -9.905469e-02
1                 audit_score -1.070906e-02
2                    drug_mar -1.734723e-17
3  aca_substance_impact_count -4.094720e-02
4           sub_effort_binary -4.940695e-02
5           sub_knowwhere_rev  3.391280e-02
